# vision_test — rail calibration

Measure the rail's **homing offset** and write it into the setup's scene (`scene/core_1000.j2`,
`core: rail_cfg: offset:`). With it, every homing lands the carriage on the bench's true zero —
so the recipes' positions along the rail are right. Do it once per bench, and again after the
rail or the robot was moved.

You need the **rail calibration probe stand** (`probe_rail_calibration`), mounted on
**fixture_plate_2 hole G3** (its `hole_0` on the hole), and the real robot.

*Generated by the Dorna AI Builder from this setup; rewritten when the setup changes — unless you edit it, then it is left as you made it.*

## Setup

In [ ]:
from pathlib import Path
import importlib, pkgutil, sys

# Run from the setup's folder or its dev/ — walk up to the one holding launch.yaml.
PROJ = Path.cwd()
while not (PROJ / "launch.yaml").exists():
    PROJ = PROJ.parent
sys.path.insert(0, str(PROJ))

# The setup's own parts (components/<type>/), registered as main.py does before the scene loads.
if (PROJ / "components").is_dir():
    for m in pkgutil.walk_packages([str(PROJ / "components")], prefix="components."):
        importlib.import_module(m.name)

from workspace.bt import Bench

VIEW_PORT = 8000          # this notebook's 3D viewer — not the orchestrator's 5000, not a runtime's 5010
bench = Bench(PROJ, port=VIEW_PORT)
workspace, core, rcp, rt = bench.workspace, bench.core, bench.rcp, bench.rt

In [ ]:
# True -> simulation (no hardware).  False -> THE REAL ROBOT.
# The call prints which one you got: "simulation api enabled" / "disabled".
core.simulation(False)

## Home the rail

**Run this before any cell below moves the arm.** The carriage crawls slowly into its hard
stop, the controller raises an alarm (expected — that stall *is* the homing), then backs off
onto the encoder index and pulls off 20 mm. A fast run at the stop, or one that never backs
off, is not right: stop and check the rail settings.

`force=True` is for after `rail_cfg.offset` changed: a warm controller keeps reporting
"homed" until it is power-cycled, and the plain path then skips homing — so the new offset
would silently not apply.

In [ ]:
from contextlib import nullcontext


def home_rail(force=False, dir=-1):
    """Home the rail against its hard stop. Returns True if homed.

    force=False -> Recipe.set_axis_with_stop (short-circuits if the controller already
                   reports the axis homed)
    force=True  -> the same set_axis + set_pid + home_with_stop sequence with the is_homed
                   check skipped, so a changed rail_cfg.offset really gets applied.
    Both run inside the device layer's expect_alarms window: the stall alarm IS homing.
    """
    if not core.has_rail:
        print("scene has no rail — nothing to home")
        return True
    cfg, api, axis = core.rail_cfg, core.robot_api, core.rail_cfg["axis"]
    sim = bool(getattr(core, "_simulation_mode", False))
    print(f"rail {cfg['type']}  axis j{axis}  offset {cfg['offset']}  {'SIMULATION (stubs)' if sim else 'REAL HARDWARE'}")
    print(f"  before: is_homed={rt.is_homed(index=axis)}  j{axis}={api.joint()[axis]}")
    if not force:
        print("  set_axis_with_stop ->", rcp["robot"].set_axis_with_stop(cfg, dir=dir))
    else:
        guard = getattr(api, "expect_alarms", None)
        with (guard("rail homing (stall)") if callable(guard) else nullcontext()):
            rt.motor(1)
            api.set_axis(index=axis, usem=cfg["usem"], usee=cfg["usee"], pprm=cfg["pprm"],
                         tprm=cfg["tprm"], ppre=cfg["ppre"], tpre=cfg["tpre"])
            rt.delay(1)
            api.set_pid(index=axis, p=cfg["p"], i=cfg["i"], d=cfg["d"],
                        duration=cfg["duration"], threshold=cfg["threshold"])
            rt.delay(1)
            print("  homing — the slow crawl into the hard stop and its alarm are expected...")
            print("  home_with_stop ->", api.home_with_stop(index=axis, val=cfg["offset"], dir=dir))
    homed = bool(rt.is_homed(index=axis))
    print(f"  after:  is_homed={homed}  j{axis}={api.joint()[axis]}")
    core.update_pose()
    return homed

## 1. Home to the stop with offset 0

The carriage drives into the hard stop and that position is taken as rail zero for the
rest of this procedure.

In [ ]:
axis = core.rail_cfg["axis"]
api = core.robot_api
guard = getattr(api, "expect_alarms", None)
with (guard("rail homing (stall)") if callable(guard) else nullcontext()):
    rt.motor(1)
    print("home_with_stop ->", api.home_with_stop(index=axis, val=0, dir=-1))
core.update_pose()
print(f"j{axis} =", api.joint()[axis])

## 2. Motor off, and push the robot to the stand

The motors go off so the robot can be moved by hand. **Hold the arm** — it is not powered.
Slide the carriage back along the rail toward the stand until the robot just touches it.

In [ ]:
rt.motor(0)

## 3. Read the rail at contact

In [ ]:
core.update_pose()
rail_value = core.robot_api.joint()[core.rail_cfg["axis"]]
print("rail at contact:", round(rail_value, 2))

## 4. The offset

| Rail | Offset |
|---|---|
| 500 mm (`rail_hd_500mm`)  | −148 − rail value |
| 1000 mm (`rail_hd_1000mm`) | −198 − rail value |
| 2000 mm (`rail_hd_2000mm`) | −398 − rail value |

Example: a 500 mm rail touching at −121.7 → offset = −148 − (−121.7) = **−26.3**.

In [ ]:
BASE = {"rail_hd_500mm": -148, "rail_hd_1000mm": -198, "rail_hd_2000mm": -398}
offset = round(BASE[core.rail_cfg["type"]] - rail_value, 2)
print(core.rail_cfg["type"], "-> offset", offset)

## 5. Write it into the scene

Sets `offset:` under `core: rail_cfg:` in `scene/core_1000.j2` and shows the change. Commit it with the
setup (Versions) so the bench keeps it.

In [ ]:
import re
core_file = PROJ / "scene/core_1000.j2"
text = core_file.read_text()
block = re.search(r"(^\s+rail_cfg:\s*\n)((?:\s{4,}.*\n)*)", text, re.M)
if block is None:
    raise RuntimeError("no rail_cfg in " + str(core_file))
body = block.group(2)
if re.search(r"^\s+offset:", body, re.M):
    new_body = re.sub(r"^(\s+offset:\s*)[^#\n]*", lambda m: m.group(1) + str(offset) + " ", body, count=1, flags=re.M)
else:
    indent = re.match(r"\s*", body).group(0) if body else "    "
    new_body = f"{indent}offset: {offset}\n" + body
core_file.write_text(text[:block.start(2)] + new_body + text[block.end(2):])
print(new_body)

## 6. Apply it

The new offset is read when the scene loads: **restart the kernel and run the Setup cells
again**, then home with `force=True` (a warm controller would otherwise skip it). The
carriage now homes onto the bench's zero. Relaunch the setup in Workspace for runs.

In [ ]:
home_rail(force=True)